# MavadoClaw Heavy Compute — Kaggle GPU Notebook

Leverages **30h GPU/week free** (P100/T4) for heavy OSINT sweeps, LLM batch inference,
memory analysis, and provider benchmarking.

## Capabilities
- OSINT sweep across multiple targets in parallel
- LLM batch processing through model cascade (Cloudflare Workers AI → PandaStack → HF Spaces)
- Memory/fact storage search and analysis
- Provider response-time benchmarking
- Automated markdown report generation

In [ ]:
import os, json, time, subprocess, sys

API_URL = os.environ.get('MAVADOCLAW_API_URL', 'https://your-space.hf.space')
ADMIN_TOKEN = os.environ.get('ADMIN_TOKEN', '')
CF_WORKER_URL = os.environ.get('CF_WORKER_URL', 'https://mavadoclaw-worker.YOUR_SUBDOMAIN.workers.dev')

try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'requests'])
    import requests

try:
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'])
    import pandas as pd

HEADERS = {'X-Admin-Token': ADMIN_TOKEN, 'Content-Type': 'application/json'}

print(f'API:          {API_URL}')
print(f'CF Worker:    {CF_WORKER_URL}')
print(f'Admin Token:  {"SET" if ADMIN_TOKEN else "NOT SET"}')

In [ ]:
targets = [
    'example.com',
    'test-target.org',
    'sample-research.net',
]

osint_results = []

for target in targets:
    print(f'[*] Sweeping {target}...')
    start = time.time()
    try:
        r = requests.post(
            f'{API_URL}/api/osint',
            json={'target': target, 'deep': True},
            headers=HEADERS,
            timeout=120,
        )
        elapsed = round(time.time() - start, 2)
        data = r.json()
        data['target'] = target
        data['elapsed_s'] = elapsed
        data['status_code'] = r.status_code
        osint_results.append(data)
        print(f'    [+] {target} done in {elapsed}s — status {r.status_code}')
    except Exception as e:
        elapsed = round(time.time() - start, 2)
        osint_results.append({'target': target, 'error': str(e), 'elapsed_s': elapsed})
        print(f'    [-] {target} failed: {e}')

print(f'\nOSINT sweep complete: {len(osint_results)} targets processed')
osint_df = pd.DataFrame(osint_results)
osint_df

In [ ]:
prompts = [
    'Summarize the latest CVEs for Apache products.',
    'What are the top 5 OSINT tools for domain reconnaissance?',
    'Explain the MITRE ATT&CK framework in 3 sentences.',
    'List common cloud misconfigurations that lead to data leaks.',
    'What DNS records are most useful for passive enumeration?',
]

batch_results = []

for i, prompt in enumerate(prompts):
    print(f'[*] Batch {i+1}/{len(prompts)}: {prompt[:50]}...')
    start = time.time()
    try:
        r = requests.post(
            f'{CF_WORKER_URL}/api/chat',
            json={
                'messages': [
                    {'role': 'system', 'content': 'You are MavadoClaw, an OSINT and security research assistant.'},
                    {'role': 'user', 'content': prompt}
                ]
            },
            headers=HEADERS,
            timeout=30,
        )
        elapsed = round(time.time() - start, 2)
        data = r.json()
        batch_results.append({
            'prompt': prompt,
            'response': data.get('content', data.get('error', ''))[:500],
            'provider': data.get('provider', 'unknown'),
            'tier': data.get('tier', 'N/A'),
            'elapsed_s': elapsed,
        })
        print(f'    [+] tier={data.get("tier")} provider={data.get("provider")} in {elapsed}s')
    except Exception as e:
        elapsed = round(time.time() - start, 2)
        batch_results.append({'prompt': prompt, 'error': str(e), 'elapsed_s': elapsed})
        print(f'    [-] Failed: {e}')

print(f'\nBatch complete: {len(batch_results)} prompts processed')
batch_df = pd.DataFrame(batch_results)
batch_df

In [ ]:
search_queries = [
    'domain reconnaissance',
    'CVE vulnerability',
    'network topology',
    'email address',
    'subdomain enumeration',
]

memory_results = []

for query in search_queries:
    print(f'[*] Searching memory: "{query}"...')
    start = time.time()
    try:
        r = requests.post(
            f'{API_URL}/api/memory/search',
            json={'query': query, 'limit': 10},
            headers=HEADERS,
            timeout=30,
        )
        elapsed = round(time.time() - start, 2)
        data = r.json()
        facts = data.get('results', data.get('facts', []))
        memory_results.append({
            'query': query,
            'count': len(facts),
            'top_fact': facts[0].get('content', facts[0].get('text', ''))[:200] if facts else '',
            'elapsed_s': elapsed,
        })
        print(f'    [+] {len(facts)} results in {elapsed}s')
    except Exception as e:
        elapsed = round(time.time() - start, 2)
        memory_results.append({'query': query, 'count': 0, 'error': str(e), 'elapsed_s': elapsed})
        print(f'    [-] Failed: {e}')

print(f'\nMemory search complete: {len(memory_results)} queries executed')
memory_df = pd.DataFrame(memory_results)
memory_df

In [ ]:
provider_endpoints = [
    {'name': 'CF Worker (Edge)', 'url': f'{CF_WORKER_URL}/api/chat', 'method': 'POST'},
    {'name': 'HF Space (Direct)', 'url': f'{API_URL}/api/chat', 'method': 'POST'},
]

bench_prompt = 'In one sentence, explain what OSINT means.'
bench_iterations = 5
bench_results = []

for provider in provider_endpoints:
    print(f'[*] Benchmarking: {provider["name"]} ({bench_iterations} iterations)...')
    timings = []
    errors = 0
    for i in range(bench_iterations):
        start = time.time()
        try:
            r = requests.post(
                provider['url'],
                json={'messages': [{'role': 'user', 'content': bench_prompt}]},
                headers=HEADERS,
                timeout=30,
            )
            elapsed = round(time.time() - start, 3)
            if r.status_code == 200:
                timings.append(elapsed)
            else:
                errors += 1
        except Exception:
            errors += 1

    avg = round(sum(timings) / len(timings), 3) if timings else None
    p50 = round(sorted(timings)[len(timings) // 2], 3) if timings else None
    p95 = round(sorted(timings)[int(len(timings) * 0.95)], 3) if timings else None
    bench_results.append({
        'provider': provider['name'],
        'iterations': bench_iterations,
        'successes': len(timings),
        'errors': errors,
        'avg_s': avg,
        'p50_s': p50,
        'p95_s': p95,
    })
    status = f'avg={avg}s p50={p50}s' if avg else 'ALL FAILED'
    print(f'    [+] {provider["name"]}: {status} ({errors} errors)')

print(f'\nBenchmark complete: {len(provider_endpoints)} providers tested')
bench_df = pd.DataFrame(bench_results)
bench_df

In [ ]:
report_ts = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())

report_lines = [
    '# MavadoClaw Heavy Compute Report',
    f'Generated: {report_ts}',
    '',
    '## OSINT Sweep Results',
    '',
]

if not osint_df.empty:
    for _, row in osint_df.iterrows():
        status = 'ERROR' if 'error' in row else 'OK'
        report_lines.append(f'- **{row.get("target", "?")}** — {status} ({row.get("elapsed_s", "?")}s)')
else:
    report_lines.append('_No OSINT results collected._')

report_lines += ['', '## LLM Batch Results', '']

if not batch_df.empty:
    for _, row in batch_df.iterrows():
        provider = row.get('provider', 'N/A')
        tier = row.get('tier', 'N/A')
        elapsed = row.get('elapsed_s', '?')
        report_lines.append(f'- **{row.get("prompt", "?")[:50]}…** — tier {tier}, {provider}, {elapsed}s')
else:
    report_lines.append('_No batch results collected._')

report_lines += ['', '## Memory Search Results', '']

if not memory_df.empty:
    for _, row in memory_df.iterrows():
        report_lines.append(f'- **{row.get("query", "?")}** — {row.get("count", 0)} facts found')
else:
    report_lines.append('_No memory results collected._')

report_lines += ['', '## Provider Benchmark', '']

if not bench_df.empty:
    for _, row in bench_df.iterrows():
        report_lines.append(
            f'- **{row.get("provider", "?")}** — avg={row.get("avg_s", "?")}s, '
            f'p50={row.get("p50_s", "?")}s, errors={row.get("errors", "?")}'
        )
else:
    report_lines.append('_No benchmark results collected._')

report_lines += ['', '---', f'_Report generated by MavadoClaw Heavy Compute — {report_ts}_']

report_md = '\n'.join(report_lines)

with open('mavadoclaw_report.md', 'w') as f:
    f.write(report_md)

print(report_md)
print('\n[+] Report saved to mavoclaw_report.md')